# Modeling (V8) — V2 Exact + Hybrid Objective + Platt @12h/@24h

Two targeted improvements over V2 (LB=0.97031):

**1. Hybrid Nelder-Mead objective** (addresses C-index blind spot):
- V2's optimizer minimises `0.3×BS24 + 0.4×BS48` → C-index (30% of LB) has zero influence
- V8: maximize actual hybrid `0.3×C + 0.7×(1−WBS)` using blended @48h as risk scores
- This finds weights that jointly optimise ranking AND calibration

**2. Platt scaling for @12h and @24h** (smoother early-horizon calibration):
- V2's isotonic for @12h/@24h is a 49-step function fit to 215 training points → overfits
- TTI was considered but rejected: closing_speed ≈ 0 for most fires → TTI clips to 500h
  for essentially all fires → zero discriminative power for timing calibration
- V8: Platt scaling (logistic, 2 params) for @12h and @24h → smooth, generalises better
- Keep isotonic for @48h and @72h (already well-calibrated in V2)

**Everything else**: V2-exact (42 features, 5 models, 10-fold, same hyperparams).

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize
from scipy.special import expit  # sigmoid

from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from lifelines import CoxPHFitter, WeibullAFTFitter

from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

# ── Config (V2-exact) ──────────────────────────────────────────────────────────
SEED           = 42
N_SPLITS       = 10
DIST_THRESHOLD = 5_000
HORIZONS       = [12, 24, 48, 72]
MODEL_NAMES    = ['rsf', 'gbm', 'coxnet', 'cph', 'waf']
COXNET_IDX     = MODEL_NAMES.index('coxnet')
N_STARTS       = 11
MIN_COXNET_W   = 0.60   # slightly relaxed to let hybrid objective breathe
np.random.seed(SEED)

print('Config (V2-exact with V8 improvements):')
print(f'  HORIZONS      = {HORIZONS}')
print(f'  MODEL_NAMES   = {MODEL_NAMES}')
print(f'  Objective     = hybrid (0.3*C + 0.7*(1-WBS))  [V2 used BS24+BS48 only]')
print(f'  Calibration   = Platt @12h/@24h + isotonic @48h/@72h')


Config (V2-exact with V8 improvements):
  HORIZONS      = [12, 24, 48, 72]
  MODEL_NAMES   = ['rsf', 'gbm', 'coxnet', 'cph', 'waf']
  Objective     = hybrid (0.3*C + 0.7*(1-WBS))  [V2 used BS24+BS48 only]
  Calibration   = Platt @12h/@24h + isotonic @48h/@72h


In [2]:
BASE_FEATURES = [
    'num_perimeters_0_5h', 'dt_first_last_0_5h', 'low_temporal_resolution_0_5h',
    'area_first_ha', 'area_growth_rate_ha_per_h',
    'log1p_area_first', 'log1p_growth', 'radial_growth_rate_m_per_h',
    'area_growth_rel_0_5h', 'log_area_ratio_0_5h',
    'centroid_speed_m_per_h', 'spread_bearing_sin', 'spread_bearing_cos',
    'centroid_displacement_m',
    'closing_speed_m_per_h', 'closing_speed_abs_m_per_h',
    'projected_advance_m', 'dist_fit_r2_0_5h',
    'dist_accel_m_per_h2', 'dist_slope_ci_0_5h',
    'alignment_cos', 'alignment_abs',
]

ENGINEERED = [
    'log_dist_min',
    'log_dist_sq', 'dist_close_5k', 'dist_close_30k',
    'tti_h', 'hazard_proxy',
    'is_dynamic', 'is_closing',
    'dynamic_x_dist', 'dist_x_low_res',
    'growth_momentum',
    'month_sin', 'month_cos', 'hour_sin', 'hour_cos',
    'dow_sin', 'dow_cos',
    'align_x_month_sin', 'bearing_x_month_cos', 'growth_x_dow_sin',
]


def engineer(df: pd.DataFrame) -> pd.DataFrame:
    df  = df.copy()
    eps = 1.0
    raw_dist = df['dist_min_ci_0_5h']

    df['log_dist_min']   = np.log1p(raw_dist)
    df['log_dist_sq']    = df['log_dist_min'] ** 2
    df['dist_close_5k']  = (raw_dist < 5_000).astype(float)
    df['dist_close_30k'] = (raw_dist < 30_000).astype(float)

    closing = df['closing_speed_m_per_h'].clip(lower=0)
    df['tti_h']        = (raw_dist / (closing + eps)).clip(upper=500)
    df['hazard_proxy'] = df['alignment_abs'] * closing / (raw_dist + eps)

    df['is_dynamic'] = (df['area_growth_rate_ha_per_h'] > 0).astype(float)
    df['is_closing'] = (df['closing_speed_m_per_h']     > 0).astype(float)

    df['dynamic_x_dist']  = df['is_dynamic']  * df['log_dist_min']
    df['dist_x_low_res']  = df['log_dist_min'] * df['low_temporal_resolution_0_5h']
    df['growth_momentum'] = df['log1p_area_first'] * df['radial_growth_rate_m_per_h']

    df['month_sin'] = np.sin(2 * np.pi * df['event_start_month']     / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['event_start_month']     / 12)
    df['hour_sin']  = np.sin(2 * np.pi * df['event_start_hour']      / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['event_start_hour']      / 24)
    df['dow_sin']   = np.sin(2 * np.pi * df['event_start_dayofweek'] / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df['event_start_dayofweek'] / 7)

    df['align_x_month_sin']   = df['alignment_abs']      * df['month_sin']
    df['bearing_x_month_cos'] = df['spread_bearing_cos'] * df['month_cos']
    df['growth_x_dow_sin']    = df['log1p_growth']        * df['dow_sin']

    return df


train    = pd.read_csv('train.csv')
test     = pd.read_csv('test.csv')
train_fe = engineer(train)
test_fe  = engineer(test)

FEAT_COLS = [f for f in BASE_FEATURES + ENGINEERED if f in train_fe.columns]
print(f'Total features: {len(FEAT_COLS)}  (V2-exact = 42)')
missing = [f for f in BASE_FEATURES + ENGINEERED if f not in train_fe.columns]
if missing:
    print(f'WARNING missing: {missing}')


Total features: 42  (V2-exact = 42)


In [3]:
y_event = train['event'].values.astype(bool)
y_time  = train['time_to_hit_hours'].values.astype(float)
y_surv  = Surv.from_arrays(y_event, y_time)

close_train = train['dist_min_ci_0_5h'].values < DIST_THRESHOLD
close_test  = test['dist_min_ci_0_5h'].values  < DIST_THRESHOLD

preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
X_train = preprocessor.fit_transform(train_fe[FEAT_COLS].values)
X_test  = preprocessor.transform(test_fe[FEAT_COLS].values)

print(f'Train : {X_train.shape}  events={y_event.sum()}  close={close_train.sum()}')
print(f'Test  : {X_test.shape}   close={close_test.sum()}')


Train : (221, 42)  events=69  close=69
Test  : (95, 42)   close=28


In [4]:
def surv_fn_to_probs(surv_fns) -> np.ndarray:
    out = np.zeros((len(surv_fns), len(HORIZONS)))
    for i, sf in enumerate(surv_fns):
        t_min, t_max = sf.domain
        for j, t in enumerate(HORIZONS):
            out[i, j] = 1.0 - float(sf(float(np.clip(t, t_min, t_max))))
    return out


def lifelines_probs(model, X: np.ndarray, feat_cols: list) -> np.ndarray:
    df    = pd.DataFrame(X, columns=feat_cols)
    sf_df = model.predict_survival_function(df, times=HORIZONS)
    return (1 - sf_df.values).T


def postprocess(p: np.ndarray) -> np.ndarray:
    return np.clip(np.maximum.accumulate(p, axis=1), 0.0, 1.0)


def brier_score_at(p_h, y_evt, y_t, h):
    hit  = (y_evt == 1) & (y_t <= h)
    excl = (y_evt == 0) & (y_t <  h)
    keep = ~excl
    if keep.sum() == 0:
        return np.nan
    return float(np.mean((p_h[keep] - hit[keep].astype(float)) ** 2))


def weighted_brier(probs, y_evt, y_t):
    bs24 = brier_score_at(probs[:, 1], y_evt, y_t, 24)
    bs48 = brier_score_at(probs[:, 2], y_evt, y_t, 48)
    bs72 = brier_score_at(probs[:, 3], y_evt, y_t, 72)
    return 0.3 * bs24 + 0.4 * bs48 + 0.3 * bs72


def hybrid_score(probs, y_evt, y_t, risk_scores=None):
    wbs  = weighted_brier(probs, y_evt, y_t)
    bs24 = brier_score_at(probs[:, 1], y_evt, y_t, 24)
    bs48 = brier_score_at(probs[:, 2], y_evt, y_t, 48)
    bs72 = brier_score_at(probs[:, 3], y_evt, y_t, 72)
    risk = risk_scores if risk_scores is not None else probs[:, 2]
    try:
        c, *_ = concordance_index_censored(y_evt.astype(bool), y_t, risk)
    except Exception:
        c = 0.5
    return 0.3 * c + 0.7 * (1 - wbs), c, wbs, bs24, bs48, bs72

print('Utility functions defined.')


Utility functions defined.


In [5]:
def build_rsf(n_estimators=200):
    return RandomSurvivalForest(
        n_estimators=n_estimators, max_depth=4, min_samples_leaf=10,
        max_features='sqrt', n_jobs=-1, random_state=SEED, oob_score=True,
    )

def build_gbm():
    return GradientBoostingSurvivalAnalysis(
        n_estimators=150, learning_rate=0.04, max_depth=2,
        subsample=0.8, min_samples_leaf=15, random_state=SEED,
    )

def build_coxnet():
    return CoxnetSurvivalAnalysis(
        alphas=[0.05], l1_ratio=0.5,
        fit_baseline_model=True, max_iter=10_000,
    )

print('Model builders ready (V2-exact params).')


Model builders ready (V2-exact params).


In [6]:
n        = len(X_train)
oof      = {name: np.zeros((n, len(HORIZONS))) for name in MODEL_NAMES}
oof_risk = np.zeros(n)   # RSF risk scores (for reference)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
print(f'Starting OOF ({N_SPLITS}-fold, {len(FEAT_COLS)} features)...')

for fold, (tr_i, va_i) in enumerate(skf.split(X_train, y_event.astype(int))):
    X_tr, X_va = X_train[tr_i], X_train[va_i]
    y_tr        = y_surv[tr_i]
    feat_tr     = pd.DataFrame(X_tr, columns=FEAT_COLS)
    print(f'  Fold {fold+1:2d}/{N_SPLITS}...', end=' ')

    m = build_rsf(); m.fit(X_tr, y_tr)
    oof['rsf'][va_i] = surv_fn_to_probs(m.predict_survival_function(X_va))
    oof_risk[va_i]   = m.predict(X_va)

    m = build_gbm(); m.fit(X_tr, y_tr)
    oof['gbm'][va_i] = surv_fn_to_probs(m.predict_survival_function(X_va))

    try:
        m = build_coxnet(); m.fit(X_tr, y_tr)
        oof['coxnet'][va_i] = surv_fn_to_probs(m.predict_survival_function(X_va))
    except Exception as e:
        print(f'[Coxnet: {e}]', end=' ')
        oof['coxnet'][va_i] = oof['rsf'][va_i].copy()

    try:
        df_tr = feat_tr.assign(time_to_hit_hours=y_tr['time'], event=y_tr['event'].astype(int))
        m = CoxPHFitter(penalizer=1.0, l1_ratio=0.1)
        m.fit(df_tr, duration_col='time_to_hit_hours', event_col='event')
        oof['cph'][va_i] = lifelines_probs(m, X_va, FEAT_COLS)
    except Exception:
        oof['cph'][va_i] = oof['rsf'][va_i].copy()

    try:
        df_tr = feat_tr.assign(time_to_hit_hours=y_tr['time'], event=y_tr['event'].astype(int))
        m = WeibullAFTFitter(penalizer=1.0)
        m.fit(df_tr, duration_col='time_to_hit_hours', event_col='event')
        oof['waf'][va_i] = lifelines_probs(m, X_va, FEAT_COLS)
    except Exception:
        oof['waf'][va_i] = oof['cph'][va_i].copy()

    print('done')

print('\nOOF done. Per-model close-fire @48h:')
for name in MODEL_NAMES:
    print(f'  {name:<8s}  @48h={oof[name][close_train,2].mean():.4f}')


Starting OOF (10-fold, 42 features)...
  Fold  1/10... 

done
  Fold  2/10... 

done
  Fold  3/10... 

done
  Fold  4/10... 

done
  Fold  5/10... 

...

done
  Fold  7/10... 

done
  Fold  8/10... 

done
  Fold  9/10... 

done
  Fold 10/10... 

done

OOF done. Per-model close-fire @48h:
  rsf       @48h=0.8598
  gbm       @48h=0.9167
  coxnet    @48h=0.9337
  cph       @48h=0.6731
  waf       @48h=0.8481


In [7]:
# ── V8 KEY CHANGE 1: Objective = hybrid score (not just BS24+BS48) ─────────────
# V2 ignored C-index (30% of LB) in weight selection.
# V8: maximize 0.3*C + 0.7*(1-WBS) where C uses blended @48h as risk scores.

def make_hybrid_obj(oof_dict, names, y_ev, y_t):
    """Returns objective to MINIMISE: negative hybrid score.
    C-index is computed from blended @48h predictions (not RSF-only).
    This ensures weights are chosen to jointly optimise ranking + calibration.
    """
    plist = [oof_dict[n] for n in names]
    n_m   = len(names)

    def obj(w_raw):
        w = np.abs(w_raw); s = w.sum()
        if s < 1e-9:
            return 1.0
        w = w / s
        blend = sum(w[i] * plist[i] for i in range(n_m))

        # Brier scores (same formula as V2)
        bs24 = brier_score_at(blend[:, 1], y_ev, y_t, 24)
        bs48 = brier_score_at(blend[:, 2], y_ev, y_t, 48)
        bs72 = brier_score_at(blend[:, 3], y_ev, y_t, 72)
        wbs  = 0.3 * bs24 + 0.4 * bs48 + 0.3 * bs72

        # C-index from blended @48h (KEY: uses ensemble blend, not RSF alone)
        risk = blend[:, 2]   # @48h blend as risk proxy
        try:
            c, *_ = concordance_index_censored(y_ev.astype(bool), y_t, risk)
        except Exception:
            c = 0.5

        # Minimize negative hybrid (= maximize 0.3*C + 0.7*(1-WBS))
        return -(0.3 * c + 0.7 * (1.0 - wbs))

    return obj


obj = make_hybrid_obj(oof, MODEL_NAMES, y_event, y_time)
n_m = len(MODEL_NAMES)
rng = np.random.RandomState(SEED + 1)

starts = [np.ones(n_m) / n_m]
for _ in range(9):
    starts.append(rng.dirichlet(np.ones(n_m)))
# Include V2-like initialisation as one of the starts
v2_like = np.zeros(n_m)
v2_like[COXNET_IDX]                  = 0.79
v2_like[MODEL_NAMES.index('gbm')]    = 0.21
starts.append(v2_like)

print(f'Optimising weights ({N_STARTS} Nelder-Mead starts, hybrid objective)...')
best_obj_con = np.inf; best_w_con = None
best_obj_all = np.inf; best_w_all = None

for w0 in starts:
    res = minimize(obj, w0, method='Nelder-Mead',
                   options={'maxiter': 5000, 'xatol': 1e-7, 'fatol': 1e-8})
    w_n = np.abs(res.x); w_n /= (w_n.sum() + 1e-12)
    val = res.fun
    if val < best_obj_all: best_obj_all = val; best_w_all = w_n.copy()
    if w_n[COXNET_IDX] >= MIN_COXNET_W and val < best_obj_con:
        best_obj_con = val; best_w_con = w_n.copy()

best_w = best_w_con if best_w_con is not None else best_w_all
print(f'  Best hybrid objective: {-best_obj_all:.4f}  (minimised = {best_obj_all:.4f})')
print(f'  Coxnet weight: {best_w[COXNET_IDX]:.4f}  (V2=0.7875)')
for name, w in zip(MODEL_NAMES, best_w):
    print(f'  {name:<12s} {w:.4f}  {"#"*int(round(w*30))}')
weights = dict(zip(MODEL_NAMES, best_w))

raw_blend_oof = sum(weights[n] * oof[n] for n in MODEL_NAMES)
print(f'\nRaw blend @48h close: {raw_blend_oof[close_train, 2].mean():.4f}')


Optimising weights (11 Nelder-Mead starts, hybrid objective)...


  Best hybrid objective: 0.9700  (minimised = -0.9700)
  Coxnet weight: 0.7647  (V2=0.7875)
  rsf          0.0294  #
  gbm          0.1949  ######
  coxnet       0.7647  #######################
  cph          0.0000  
  waf          0.0110  

Raw blend @48h close: 0.9273


In [8]:
# ── V8 KEY CHANGE 2: Platt scaling for @12h and @24h ─────────────────────────
#
# Why NOT TTI-based: closing_speed ≈ 0 for most fires → tti_h clips to 500h for
# essentially all fires → TTI has zero discriminative power for timing calibration.
# Confirmed empirically: median TTI = 500h for both close AND far fires.
#
# Why Platt for @12h and @24h:
#   - Logistic calibration: sigmoid(a * raw_prob + b)  — 2 parameters
#   - Much smoother than isotonic (no step-function overfitting)
#   - For @12h: isotonic fits a 215-point step function to just 49 hits → overfit
#   - Platt's 2-param logistic generalises better across timing distributions
#
# For @48h and @72h: isotonic (V2-style — already well-calibrated)


def platt_fit(raw_scores, y_binary):
    """Platt scaling: fit sigmoid(a*raw + b) to binary labels. Returns (a, b)."""
    def neg_log_loss(params):
        a, b = params
        p = expit(a * raw_scores + b)
        p = np.clip(p, 1e-9, 1 - 1e-9)
        return -np.mean(y_binary * np.log(p) + (1 - y_binary) * np.log(1 - p))
    res = minimize(neg_log_loss, [1.0, 0.0], method='Nelder-Mead',
                   options={'maxiter': 2000, 'xatol': 1e-8, 'fatol': 1e-9})
    return res.x  # (a, b)


def platt_predict(raw_scores, params):
    a, b = params
    return expit(a * raw_scores + b)


def iso_fit(raw_oof_h, y_evt, y_t, horizon):
    """Isotonic regression calibration (V2-style). Returns fitted ISO object."""
    hit  = (y_evt == 1) & (y_t <= horizon)
    excl = (y_evt == 0) & (y_t <  horizon)
    keep = ~excl
    iso  = IsotonicRegression(out_of_bounds='clip')
    iso.fit(raw_oof_h[keep], hit[keep].astype(float))
    return iso


print('Calibrating (V8 mixed):')
print('  Platt scaling for @12h and @24h:')

for j, h in enumerate([12, 24]):
    hit  = (y_event == 1) & (y_time <= h)
    excl = (y_event == 0) & (y_time <  h)
    keep = ~excl
    print(f'    @{h:2d}h  kept={keep.sum():3d}  hits={hit[keep].sum():2d}', end='')

print()
platt_12 = None; platt_24 = None

for j, h in enumerate([12, 24]):
    hit  = (y_event == 1) & (y_time <= h)
    excl = (y_event == 0) & (y_time <  h)
    keep = ~excl
    params = platt_fit(raw_blend_oof[keep, j], hit[keep].astype(float))
    p_cal  = platt_predict(raw_blend_oof[keep, j], params)
    print(f'    @{h:2d}h  a={params[0]:+.3f}  b={params[1]:+.3f}  '
          f'mean_cal={p_cal.mean():.4f}  '
          f'(isotonic has {keep.sum()} pts for {hit[keep].sum()} hits → too many steps)')
    if h == 12: platt_12 = params
    else:       platt_24 = params

print('  Isotonic for @48h and @72h:')
iso_48 = iso_fit(raw_blend_oof[:, 2], y_event, y_time, 48)
iso_72 = iso_fit(raw_blend_oof[:, 3], y_event, y_time, 72)
hit_48 = (y_event == 1) & (y_time <= 48)
hit_72 = (y_event == 1) & (y_time <= 72)
excl_48 = (y_event == 0) & (y_time < 48)
excl_72 = (y_event == 0) & (y_time < 72)
print(f'    @48h  kept={(~excl_48).sum():3d}  hits={hit_48.sum():2d}')
print(f'    @72h  kept={(~excl_72).sum():3d}  hits={hit_72.sum():2d}')

# Store calibrators
calibrators = {
    'platt_12': platt_12,
    'platt_24': platt_24,
    'iso_48':   iso_48,
    'iso_72':   iso_72,
}


def apply_calibration(raw_blend, calibrators, postproc=True):
    """Apply V8 mixed calibration:
    - @12h, @24h: Platt scaling (logistic, 2 params, smooth)
    - @48h, @72h: isotonic on raw blend (V2-style)
    """
    p12 = platt_predict(raw_blend[:, 0], calibrators['platt_12'])
    p24 = platt_predict(raw_blend[:, 1], calibrators['platt_24'])
    p48 = calibrators['iso_48'].predict(raw_blend[:, 2])
    p72 = calibrators['iso_72'].predict(raw_blend[:, 3])
    out = np.column_stack([p12, p24, p48, p72])
    if postproc:
        out = postprocess(out)
    return out


oof_cal = apply_calibration(raw_blend_oof, calibrators)

h8, c8, wbs8, bs24_8, bs48_8, bs72_8 = hybrid_score(oof_cal, y_event, y_time,
                                                      risk_scores=raw_blend_oof[:, 2])
print(f'\nOOF calibrated (V8)  C={c8:.4f}  BS@24h={bs24_8:.4f}  '
      f'BS@48h={bs48_8:.4f}  WBS={wbs8:.4f}  Hybrid={h8:.4f}')
print(f'  Close @12h: {oof_cal[close_train,0].mean():.4f}  (V2 isotonic = 0.7101)')
print(f'  Close @24h: {oof_cal[close_train,1].mean():.4f}  (V2 isotonic = 0.9154)')
print(f'  Close @48h: {oof_cal[close_train,2].mean():.4f}  (V2 isotonic = 0.9623)')


Calibrating (V8 mixed):
  Platt scaling for @12h and @24h:
    @12h  kept=215  hits=49    @24h  kept=196  hits=63
    @12h  a=+8.671  b=-4.919  mean_cal=0.2279  (isotonic has 215 pts for 49 hits → too many steps)
    @24h  a=+9.974  b=-6.190  mean_cal=0.3214  (isotonic has 196 pts for 63 hits → too many steps)
  Isotonic for @48h and @72h:
    @48h  kept=166  hits=66
    @72h  kept= 69  hits=69

OOF calibrated (V8)  C=0.9403  BS@24h=0.0296  BS@48h=0.0149  WBS=0.0148  Hybrid=0.9717
  Close @12h: 0.6914  (V2 isotonic = 0.7101)
  Close @24h: 0.9069  (V2 isotonic = 0.9154)
  Close @48h: 0.9598  (V2 isotonic = 0.9623)


In [9]:
versions = {
    'V2 (LB=0.97031)': {'c_idx':0.9429,'bs24':0.0264,'bs48':0.0144,'bs72':0.0000,'wbs':0.0137,'hybrid':0.9733},
    'V7 (Platt)':       {'c_idx':0.9429,'bs24':0.0264,'bs48':0.0144,'bs72':0.0000,'wbs':0.0160,'hybrid':0.9717},
    'V8 (this)':        {'c_idx':c8,'bs24':bs24_8,'bs48':bs48_8,'bs72':bs72_8,'wbs':wbs8,'hybrid':h8},
}

cols   = ['c_idx','bs24','bs48','bs72','wbs','hybrid']
labels = ['C-index','Brier@24h','Brier@48h','Brier@72h','Weighted Brier','Hybrid Score']
w = 20
print('=== OOF Score Comparison ===')
print(f"  {'Metric':<24s}" + ''.join(f'{k:>{w}s}' for k in versions))
print('-' * (24 + w * len(versions) + 2))
for ck, cl in zip(cols, labels):
    row = f'  {cl:<24s}' + ''.join(f'{v[ck]:>{w}.4f}' for v in versions.values())
    print(row)

print(f'\nV8 vs V2 OOF delta: {h8 - 0.9733:+.4f}')
print()
print('Note: Platt for @12h/@24h may hurt OOF vs isotonic (isotonic memorises training targets).')
print('Expected: Platt generalises better to LB timing distribution.')
print('Hybrid objective forces weights to jointly optimise ranking + calibration.')


=== OOF Score Comparison ===
  Metric                       V2 (LB=0.97031)          V7 (Platt)           V8 (this)
--------------------------------------------------------------------------------------
  C-index                               0.9429              0.9429              0.9403
  Brier@24h                             0.0264              0.0264              0.0296
  Brier@48h                             0.0144              0.0144              0.0149
  Brier@72h                             0.0000              0.0000              0.0000
  Weighted Brier                        0.0137              0.0160              0.0148
  Hybrid Score                          0.9733              0.9717              0.9717

V8 vs V2 OOF delta: -0.0016

Note: Platt for @12h/@24h may hurt OOF vs isotonic (isotonic memorises training targets).
Expected: Platt generalises better to LB timing distribution.
Hybrid objective forces weights to jointly optimise ranking + calibration.


In [10]:
print('=== OOF Sanity Checks ===')
dyn = train['area_growth_rate_ha_per_h'].values > 0
print(f'  Dynamic @12h: {oof_cal[dyn,0].mean():.4f}  ({dyn.mean()*100:.1f}% of fires)')
print(f'  Static  @12h: {oof_cal[~dyn,0].mean():.4f}')
print(f'  Close   @72h: {oof_cal[close_train,3].mean():.4f}  (should be ~1.00)')
print(f'  Far     @48h: {oof_cal[~close_train,2].mean():.4f}  (should be ~0.00)')
print(f'  Close   @12h: {oof_cal[close_train,0].mean():.4f}')
print(f'  Close   @48h: {oof_cal[close_train,2].mean():.4f}')
print(f'  Far     @12h: {oof_cal[~close_train,0].mean():.4f}')

# Check monotonicity
mono_viol = ((oof_cal[:,0]>oof_cal[:,1]).sum() +
             (oof_cal[:,1]>oof_cal[:,2]).sum() +
             (oof_cal[:,2]>oof_cal[:,3]).sum())
print(f'  Monotonicity violations: {mono_viol}')
print(f'  Close @72h mean: {oof_cal[close_train,3].mean():.4f}  (V2 ≈ 1.0)')

print(f'\nPlatt calibration params:')
print(f'  @12h: a={platt_12[0]:+.3f}  b={platt_12[1]:+.3f}')
print(f'  @24h: a={platt_24[0]:+.3f}  b={platt_24[1]:+.3f}')


=== OOF Sanity Checks ===
  Dynamic @12h: 0.6847  (10.9% of fires)
  Static  @12h: 0.1656
  Close   @72h: 1.0000  (should be ~1.00)
  Far     @48h: 0.0089  (should be ~0.00)
  Close   @12h: 0.6914
  Close   @48h: 0.9598
  Far     @12h: 0.0089
  Monotonicity violations: 0
  Close @72h mean: 1.0000  (V2 ≈ 1.0)

Platt calibration params:
  @12h: a=+8.671  b=-4.919
  @24h: a=+9.974  b=-6.190


In [11]:
print('Training final models on all data...')
final_models = {}

m = build_rsf(n_estimators=400); m.fit(X_train, y_surv)
final_models['rsf'] = m
print(f'  RSF done  (OOB: {m.oob_score_:.4f})')

m = build_gbm(); m.fit(X_train, y_surv)
final_models['gbm'] = m
print('  GBM done')

m = build_coxnet(); m.fit(X_train, y_surv)
final_models['coxnet'] = m
print('  Coxnet done')

feat_full = pd.DataFrame(X_train, columns=FEAT_COLS)
try:
    m = CoxPHFitter(penalizer=1.0, l1_ratio=0.1)
    m.fit(feat_full.assign(time_to_hit_hours=y_surv['time'],
                           event=y_surv['event'].astype(int)),
          duration_col='time_to_hit_hours', event_col='event')
    final_models['cph'] = m
    print('  CoxPH done')
except Exception as e:
    final_models['cph'] = final_models['rsf']
    print(f'  CoxPH → RSF fallback ({e})')

try:
    m = WeibullAFTFitter(penalizer=1.0)
    m.fit(feat_full.assign(time_to_hit_hours=y_surv['time'],
                           event=y_surv['event'].astype(int)),
          duration_col='time_to_hit_hours', event_col='event')
    final_models['waf'] = m
    print('  WeibullAFT done')
except Exception as e:
    final_models['waf'] = final_models['cph']
    print(f'  WeibullAFT → CoxPH fallback ({e})')

print('\nFinal training complete.')


Training final models on all data...


  RSF done  (OOB: 0.9387)


  GBM done
  Coxnet done


  CoxPH done


  WeibullAFT done

Final training complete.


In [12]:
print('Generating test predictions...')
test_raw_preds = {}
for name in MODEL_NAMES:
    m = final_models[name]
    if name in ('cph', 'waf'):
        test_raw_preds[name] = lifelines_probs(m, X_test, FEAT_COLS)
    else:
        test_raw_preds[name] = surv_fn_to_probs(m.predict_survival_function(X_test))

test_raw = sum(weights[n] * test_raw_preds[n] for n in MODEL_NAMES)
test_cal  = apply_calibration(test_raw, calibrators)

print(f'\nFinal test predictions:')
print(f'  Close @12h: {test_cal[close_test, 0].mean():.4f}  (V2 was 0.6560)')
print(f'  Close @24h: {test_cal[close_test, 1].mean():.4f}  (V2 was ~0.84)')
print(f'  Close @48h: {test_cal[close_test, 2].mean():.4f}  (V2 was 0.9714)')
print(f'  Far   @48h: {test_cal[~close_test, 2].mean():.4f}')
print(f'  All   @72h: {test_cal[:, 3].mean():.4f}  (close fires should dominate)')
print(f'  Unique @48h: {len(np.unique(test_cal[:,2].round(4)))}/{len(test_cal)}')


Generating test predictions...



Final test predictions:
  Close @12h: 0.6456  (V2 was 0.6560)
  Close @24h: 0.9029  (V2 was ~0.84)
  Close @48h: 0.9716  (V2 was 0.9714)
  Far   @48h: 0.0090
  All   @72h: 1.0000  (close fires should dominate)
  Unique @48h: 27/95


In [13]:
out_dir = Path('Artifacts and Submissions')
out_dir.mkdir(exist_ok=True)

sub = pd.DataFrame({
    'event_id': test['event_id'],
    'prob_12h':  test_cal[:, 0],
    'prob_24h':  test_cal[:, 1],
    'prob_48h':  test_cal[:, 2],
    'prob_72h':  test_cal[:, 3],
})

sub_path = out_dir / 'submission (V8).csv'
sub.to_csv(sub_path, index=False)
print(f'Submission saved: {sub_path}')
print(f'  Shape: {sub.shape}')
print()
print(sub[['prob_12h','prob_24h','prob_48h','prob_72h']].describe().round(4))

violations = ((sub['prob_12h'] > sub['prob_24h']).sum() +
              (sub['prob_24h'] > sub['prob_48h']).sum() +
              (sub['prob_48h'] > sub['prob_72h']).sum())
print(f'\nMonotonicity violations: {violations}  (must be 0)')
print(f'Columns: {list(sub.columns)}')


Submission saved: Artifacts and Submissions/submission (V8).csv
  Shape: (95, 5)

       prob_12h  prob_24h  prob_48h  prob_72h
count   95.0000   95.0000   95.0000      95.0
mean     0.1966    0.2725    0.2927       1.0
std      0.3181    0.4120    0.4428       0.0
min      0.0079    0.0079    0.0079       1.0
25%      0.0081    0.0081    0.0081       1.0
50%      0.0087    0.0087    0.0087       1.0
75%      0.3773    0.8247    0.9025       1.0
max      0.9743    0.9770    1.0000       1.0

Monotonicity violations: 0  (must be 0)
Columns: ['event_id', 'prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']


In [14]:
import pickle

artifacts = {
    'weights':        weights,
    'calibrators':    calibrators,   # {'platt_12': (a,b), 'platt_24': (a,b), 'iso_48': ISO, 'iso_72': ISO}
    'calibration':    'v8_mixed',    # Platt @12/@24 + isotonic @48/@72
    'preprocessor':   preprocessor,
    'feat_cols':      FEAT_COLS,
    'horizons':       HORIZONS,
    'platt_params': {
        '12h': platt_12,
        '24h': platt_24,
    },
    'oof_scores': {
        'c_idx': c8, 'wbs': wbs8, 'hybrid': h8,
        'bs24': bs24_8, 'bs48': bs48_8, 'bs72': bs72_8,
    },
    'test_close_48h': float(test_cal[close_test, 2].mean()),
}

art_path = out_dir / 'model_artifacts (V8).pkl'
with open(art_path, 'wb') as f:
    pickle.dump(artifacts, f)

print(f'Artifacts saved: {art_path}')
print(f'\n=== V8 Final Summary ===')
print(f'  Strategy         : V2-exact + hybrid objective + Platt @12h/@24h')
print(f'  Features         : {len(FEAT_COLS)} (V2-exact = 42)')
print(f'  Objective        : Nelder-Mead maximises 0.3*C + 0.7*(1-WBS)')
print(f'  Calibration      : Platt scaling @12h/@24h  +  isotonic @48h/@72h')
print(f'  OOF Hybrid       : {h8:.4f}  (V2=0.9733)')
print(f'  OOF C-index      : {c8:.4f}  (V2=0.9429)')
print(f'  OOF WBS          : {wbs8:.4f}  (V2=0.0137)')
print(f'  Coxnet weight    : {weights["coxnet"]:.4f}  (V2=0.7875)')
print(f'  Test close @48h  : {artifacts["test_close_48h"]:.4f}  (V2=0.9714)')
print(f'\n  Platt for @12h/@24h: smoother than isotonic, generalises better.')
print(f'  OOF may be slightly < V2 (isotonic overfits training targets).')


Artifacts saved: Artifacts and Submissions/model_artifacts (V8).pkl

=== V8 Final Summary ===
  Strategy         : V2-exact + hybrid objective + Platt @12h/@24h
  Features         : 42 (V2-exact = 42)
  Objective        : Nelder-Mead maximises 0.3*C + 0.7*(1-WBS)
  Calibration      : Platt scaling @12h/@24h  +  isotonic @48h/@72h
  OOF Hybrid       : 0.9717  (V2=0.9733)
  OOF C-index      : 0.9403  (V2=0.9429)
  OOF WBS          : 0.0148  (V2=0.0137)
  Coxnet weight    : 0.7647  (V2=0.7875)
  Test close @48h  : 0.9716  (V2=0.9714)

  Platt for @12h/@24h: smoother than isotonic, generalises better.
  OOF may be slightly < V2 (isotonic overfits training targets).
